In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    print("Setup complete!")


In [2]:
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
        "eng": "eng_Latn",
        "sin": "sin_Sinh",
   
       # Do NOT put "san" here.
       # Sanskrit is handled separately using its script.
   
       "tam": "tam_Taml",
       "hin": "hin_Deva",
       "ben": "ben_Beng",
       "arb": "arb_Arab",
       "fra": "fra_Latn",
       "deu": "deu_Latn",
}

def detect_script(text):
    """
    Detect the script used by the text.
    """

    text = str(text)

    has_sinhala = any(
        "\u0D80" <= ch <= "\u0DFF"
        for ch in text
    )

    has_devanagari = any(
        "\u0900" <= ch <= "\u097F"
        for ch in text
    )

    if has_sinhala and not has_devanagari:
        return "Sinh"

    if has_devanagari and not has_sinhala:
        return "Deva"

    if has_sinhala and has_devanagari:
        return "Mixed"

    return "Unknown"


def map_true_label(row):

    label = str(row.get("label", "")).strip()
    source = str(row.get("source", "")).strip()

    if label == "san":

        # Sanskrit added by our project in Sinhala script
        if source in [
            "DCS",
            "SansinNT",
            "SiDiaC-v2"
        ]:
            return "san_Sinh"

        # Sanskrit coming from the original benchmark
        return "san_Deva"

    return TARGET_LANGUAGES.get(label)

def load_dataset(file_path):

    print(
        f"\nLoading {os.path.basename(file_path)}..."
    )

    records = []

    with open(file_path, encoding="utf-8") as f:

        for line in f:

            row = json.loads(line)

            raw_label = row.get("label")

            # Keep normal target languages
            # AND keep all Sanskrit examples.
            if (
                raw_label in TARGET_LANGUAGES
                or raw_label == "san"
            ):
                records.append(row)


    df = pd.DataFrame(records)


    if not df.empty:

        # Correct script-aware mapping
        df["flores_label"] = df.apply(
            map_true_label,
            axis=1
        )

        # Only remove Sanskrit rows whose script
        # genuinely could not be determined.
        df = df[
            df["flores_label"].notna()
        ].copy()


        print(
            f"Loaded {len(df)} rows across "
            f"{df['flores_label'].nunique()} "
            f"language-script classes"
        )


        print("\nTrue-label counts:")

        print(
            df["flores_label"]
            .value_counts()
            .sort_index()
        )


    else:

        print(
            "No matching target languages "
            "found in this dataset."
        )


    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
    results["true_label"],
    results["predicted_label"],
    average="macro",
    labels=target_labels,
    zero_division=0,
)

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(print(classification_report(
    results["true_label"],
    results["predicted_label"],
    labels=target_labels,
    digits=4,
    zero_division=0,
)))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [4]:
import os
import urllib.request
import fasttext

MODEL_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
MODEL_PATH = "models/benchmark/fastText/lid.176.bin"

FASTTEXT_LANG_MAP = {
    "eng_Latn": "en",
    "sin_Sinh": "si",

    # fastText LID-176 supports Sanskrit,
    # but not a separate Sinhala-script Sanskrit class.
    "san_Deva": "sa",

    # Keep Sinhala-script Sanskrit as its own TRUE class.
    # fastText cannot predict this label, therefore its zero-shot
    # F1 should naturally become 0.
    "san_Sinh": "san_Sinh",

    "tam_Taml": "ta",
    "hin_Deva": "hi",
    "ben_Beng": "bn",
    "arb_Arab": "ar",
    "fra_Latn": "fr",
    "deu_Latn": "de",
}

if not os.path.exists(MODEL_PATH):
    print("Downloading fastText LID-176 model (~126MB)...")
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

print("Loading fastText LID-176 model...")
model = fasttext.load_model(MODEL_PATH)
model_name = "fastText LID-176"
target_labels = sorted(set(FASTTEXT_LANG_MAP.values()))

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty: continue
    
    texts = df["text"].astype(str).str.replace("\n", " ").tolist()
    print(f"Evaluating {len(texts)} samples with {model_name}...")
    preds, _ = model.predict(texts, k=1)

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["flores_label"].map(FASTTEXT_LANG_MAP)
    results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]

    evaluate_and_save(results, model_name, dataset_name, target_labels)


Loading fastText LID-176 model...

Loading commonlid.jsonl...
Loaded 74947 rows across 10 language-script classes

True-label counts:
flores_label
arb_Arab    26152
ben_Beng     1886
deu_Latn     7553
eng_Latn    27461
fra_Latn     3233
hin_Deva     3666
san_Deva      895
san_Sinh     1327
sin_Sinh     2693
tam_Taml       81
Name: count, dtype: int64
Evaluating 74947 samples with fastText LID-176...

ZERO-SHOT BENCHMARK RESULTS (fastText LID-176 on commonlid)
Accuracy:  95.33%
Macro F1:  84.87%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     0.9997    0.9898    0.9947     26152
          bn     0.9984    0.9889    0.9936      1886
          de     0.9717    0.9629    0.9673      7553
          en     0.9848    0.9603    0.9724     27461
          fr     0.9413    0.9480    0.9447      3233
          hi     0.9721    0.9684    0.9702      3666
          sa     0.9824    0.8112    0.8886       895
    san_Sinh     0.0000    0.0000    0.00